# WSW Drift Checker

Iterate through folders, reporting clock drift.

Copyright 2026 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

## Notes

This is very crude and simply looks for Doppler speeds in excess of 30 m/s (i.e. 108 km/h or 58.32 knots).

Such speeds are currently unrealistic for a windsurfer.

In [16]:
import os
import sys

import numpy as np

from shapely.geometry import Polygon, Point

import traceback

corePath = os.path.join('..', 'core')
if corePath not in sys.path:
    sys.path.extend([corePath])

from file_reader import getFileReader

## Main Function

In [17]:
def findSpikes():
    '''Iterate through session archive testing each GPS file'''

    rootDir = os.path.join(projdir, '..', 'wsw-data', 'events', '2019')

    errors = {}
    
    totFiles = 0
    totPoints = 0
    totDist = 0

    for root, subDirs, files in sorted(os.walk(rootDir)):
        for file in sorted(files):
            ext = os.path.splitext(file)[1].lower()

            if ext in ['.sbn']:
                filePath = os.path.join(root, file)
                reader = getFileReader(filePath)
                try:
                    reader.load()
                    
                    # Process all tracks within the file
                    for track in reader.tracks:

                        # Select points where number of sats is not zero
                        points = (np.argwhere(track.data['sat'] > 0))
                        drift = np.median(track.data['clock_drift'][points])

                        print(os.path.basename(root), file, reader.device, int(drift))
                            
                        totPoints += len(track.data['sog'])
                        if ext in ['.oao']:
                            totDist += track.data['sog'].sum() / 5
                        else:
                            totDist += track.data['sog'].sum()

                    totFiles += 1

                except Exception:
                    errors[filePath.replace(projdir + '/', '')] = traceback.format_exc()

    print(os.linesep * 2 + 'Summary: files = {:,}, points = {:,}, distance = {:,.02f} km'.format(
        totFiles, totPoints, totDist / 1000))

    if len(errors) > 0:
        print(os.linesep * 2 + 'Errors:')
        for error in errors:
            print(error)
            print(errors[error])

In [18]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    findSpikes()
    
    print(os.linesep + 'All done!')

20191005 FALEY40JAMES_932000563_20191005_093834.SBN GT-31 1848170
20191005 GEORG5MICHA_123201112_20191005_134818.SBN GT-31 1819246
20191005 LONGM42JAMES_113200508_20191005_121930.SBN GT-31 1822062
20191005 STALL39SCOTT_832001487_20191005_150800.SBN GT-31 1801624
20191005 WILLI37DAVID_932000560_20191005_130118.SBN GT-31 1821129
20191006 ARNOL20SPENS_932001083_20191006_103752.SBN GT-31 1817095
20191006 BAKER24TYLER_833002174_20191006_101124.SBN GT-31 1840121
20191006 BAKER24TYLER_932000571_20191006_101124.SBN GT-31 1840121
20191006 BATTY3JAMES_103200479_20191006_084900.SBN GT-31 1754546
20191006 CARTR18BOB_932000712_20191006_095218.SBN GT-31 1839588
20191006 CLELA51CHRIS_113200978_20191006_084825.SBN GT-31 1823784
20191006 CONNE2GARRY_133200827_20191006_101137.SBN GT-31 1820996
20191006 CROSB25RICHA_932000574_20191006_095048.SBN GT-31 1841072
20191006 CROSB47PETER_933000012_20191006_092530.SBN GT-31 1823965
20191006 CROSS55JIM_133200501_20191006_103148.SBN GT-31 1820540
20191006 DANNY19C